# Capstone project: Personal Expense Tracker
* congratulations for making it all the way here - this is the LAST lesson of the whole course
* instead of learning something brand new, we are going to build one small but complete project that ties together a bunch of the concepts from the previous 17 lessons
* the project: a "Personal Expense Tracker" - something that lets us add expenses, group/summarize them, save them to a real database file and load them back
* this is intentionally small and simple, real world apps obviously get way more complex, but the goal here is to SEE how all the pieces we learned fit together into one coherent thing, not to build the next big billion dollar startup
* we will build this step by step, recapping the concept we reuse in each step (with a link back to the lesson number where we first learned it)

## imports we need
* we only use the standard library here - no pip installs, this whole course stayed dependency-free on purpose so you can run it literally anywhere python is installed

In [1]:
# dataclasses -> for our Expense record (recap lesson 10, typing)
from dataclasses import dataclass, field

# sqlite3 -> for saving/loading our expenses to a real (tiny) database file (recap lesson 13)
import sqlite3

# os -> we use this at the very end to delete the demo database file again, we dont want to leave junk files in the repo
import os

# collections.defaultdict -> a dict that auto-creates missing keys with a default value, super handy for "group and sum" style code (recap lesson 12)
from collections import defaultdict

# functools.wraps -> keeps a decorated functions name/docstring intact, we use this for our timing decorator (recap lesson 09)
from functools import wraps

# time -> we use time.perf_counter() to measure how long generating our report takes
import time

# logging -> instead of a plain print() for our timing info, we use the logging module like a "real" application would (recap lesson 15)
import logging

# unittest -> the standard library testing framework, we write a couple of tests for our tracker (recap lesson 15)
import unittest

# typing helpers -> Iterator is the type hint for "a function that yields values instead of returning one value", Optional means "this or None"
from typing import Iterator, Optional

# a very basic logging setup so our timing decorator has somewhere to print its messages to (this just goes to the console/notebook output)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

print("all imports ready, lets build the tracker!")

all imports ready, lets build the tracker!


## step 1: the Expense record (recap dataclasses, lesson 10)
* a dataclass is the perfect fit for a simple "just holds some data" object like a single expense
* it auto-generates `__init__`, `__repr__` and `__eq__` for us, so we dont need to write that boilerplate by hand
* we add type hints to every field - remember from lesson 10: these hints are DOCUMENTATION ONLY, python does not enforce them at runtime by itself (more on that later in this notebook again)

In [2]:
@dataclass
class Expense:
    """A single expense entry. Just a plain data container, no behaviour attached."""
    amount: float        # how much money was spent, we store it as a float (e.g. 12.5)
    category: str        # a free-text category like "food", "transport", "rent", ...
    description: str     # a short human-readable note about what this expense was for
    date: str            # we keep it simple and store the date as an ISO formatted string, e.g. "2024-03-01"

# lets create one manually to see what we get for free from @dataclass
first_expense = Expense(amount=12.50, category="food", description="lunch at the office", date="2024-03-01")
print(first_expense)                 # a nice auto-generated __repr__, way better than the default "<Expense object at 0x...>"
print(first_expense.amount)          # normal attribute access, just like any other object
print(first_expense == Expense(12.50, "food", "lunch at the office", "2024-03-01"))  # auto-generated __eq__ compares field by field

Expense(amount=12.5, category='food', description='lunch at the office', date='2024-03-01')
12.5
True


## step 2: a custom exception (recap error handling, lesson 06)
* we dont want to allow nonsense expenses, like a negative amount or an empty category
* instead of raising a generic `ValueError` everywhere, we define our own exception type - this lets any code that USES our tracker catch exactly this kind of error, separately from other unrelated bugs

In [3]:
class InvalidExpenseError(Exception):
    """Raised when someone tries to create an expense that fails our validation rules."""
    pass  # we dont need any extra behaviour, inheriting from Exception is enough to get a fully working exception type

# a tiny standalone demo before we wire it into the tracker class
def validate_amount(amount):
    if amount <= 0:
        # raising our OWN exception type instead of a generic ValueError - this is the whole point of a custom exception
        raise InvalidExpenseError(f"amount must be positive, got {amount}")
    return amount

try:
    validate_amount(-5)
except InvalidExpenseError as error:
    print("caught our custom exception:", error)

caught our custom exception: amount must be positive, got -5


## step 3: the ExpenseTracker class (recap OOP, lessons 07/08)
* now we wrap everything in a class that owns a list of `Expense` objects and offers methods to work with them
* `add_expense` does the validation and raises `InvalidExpenseError` on bad input (this is where step 2 gets used for real)
* `summary_by_category` uses a `defaultdict(float)` to group and sum amounts per category without needing an "if category not in dict" check every time (recap lesson 12)

In [4]:
class ExpenseTracker:
    """Keeps a list of Expense objects and offers ways to add, filter and summarize them."""

    def __init__(self):
        self.expenses: list[Expense] = []  # start empty, this list is our whole "database" while the program runs

    def add_expense(self, amount: float, category: str, description: str, date: str) -> Expense:
        """Validate the given values and, if they are ok, store a new Expense. Raises InvalidExpenseError otherwise."""
        if amount <= 0:
            # this is our main validation rule the whole capstone is built around
            raise InvalidExpenseError(f"amount must be positive, got {amount}")
        if not category:
            # an empty string is falsy (recap lesson 03), so this also catches None-ish/empty categories
            raise InvalidExpenseError("category cant be empty")
        expense = Expense(amount=amount, category=category, description=description, date=date)  # build the record
        self.expenses.append(expense)  # only append AFTER validation passed
        return expense  # handing the created expense back can be convenient for the caller

    def list_expenses(self) -> list[Expense]:
        """Return a (shallow) copy of all stored expenses."""
        return list(self.expenses)  # returning a copy, not the internal list itself, so callers cant accidentally mess with our internal state

    def filter_by_category(self, category: str) -> list[Expense]:
        """Return only the expenses that match the given category."""
        return [expense for expense in self.expenses if expense.category == category]  # a simple list comprehension (recap lesson 03)

    def total(self) -> float:
        """Sum up the amount of every stored expense."""
        return sum(expense.amount for expense in self.expenses)  # a generator expression fed straight into sum(), no intermediate list needed

    def summary_by_category(self) -> dict:
        """Group expenses by category and sum the amounts per category."""
        summary = defaultdict(float)  # defaultdict(float) means: a missing key silently starts at 0.0 instead of raising a KeyError
        for expense in self.expenses:
            summary[expense.category] += expense.amount  # no need to check "if category not in summary" first, defaultdict handles that for us
        return dict(summary)  # convert back to a plain dict before returning, callers shouldnt need to know we used defaultdict internally

print("ExpenseTracker class defined")

ExpenseTracker class defined


## quick smoke test of the tracker so far
* before we add persistence and reports, lets just try the class out with a couple of expenses to make sure the basics work

In [5]:
demo_tracker = ExpenseTracker()  # a throwaway tracker just for this quick smoke test

# add a handful of expenses across a couple of categories
demo_tracker.add_expense(12.50, "food", "lunch", "2024-03-01")
demo_tracker.add_expense(45.00, "transport", "monthly train ticket", "2024-03-01")
demo_tracker.add_expense(7.90, "food", "coffee and a snack", "2024-03-02")

print("all expenses:", demo_tracker.list_expenses())
print("food only:", demo_tracker.filter_by_category("food"))
print("total spent:", demo_tracker.total())
print("summary by category:", demo_tracker.summary_by_category())

# and now lets actually trigger our custom exception on purpose, to see it in action
try:
    demo_tracker.add_expense(-10, "food", "this should never be allowed", "2024-03-03")
except InvalidExpenseError as error:
    print("caught it as expected:", error)

all expenses: [Expense(amount=12.5, category='food', description='lunch', date='2024-03-01'), Expense(amount=45.0, category='transport', description='monthly train ticket', date='2024-03-01'), Expense(amount=7.9, category='food', description='coffee and a snack', date='2024-03-02')]
food only: [Expense(amount=12.5, category='food', description='lunch', date='2024-03-01'), Expense(amount=7.9, category='food', description='coffee and a snack', date='2024-03-02')]
total spent: 65.4
summary by category: {'food': 20.4, 'transport': 45.0}
caught it as expected: amount must be positive, got -10


## step 4: persistence with sqlite3 (recap databases, lesson 13)
* keeping expenses only in memory means they are gone the second the program ends, not very useful for a REAL expense tracker
* sqlite3 is part of the python standard library and stores an entire database in a single file - perfect for a small project like this, no server setup needed
* we add a `save()` and a `load()` method directly on `ExpenseTracker`, both create the table if it doesnt exist yet, so calling them on a brand new database file just works

In [6]:
# the filename of our little demo database - we clean this file up again at the very end of the notebook
DB_FILENAME = "capstone_expense_tracker_demo.db"

# the SQL to create our table - CREATE TABLE IF NOT EXISTS means this is safe to run every time, it wont wipe existing data
CREATE_TABLE_SQL = """
    CREATE TABLE IF NOT EXISTS expenses (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        amount REAL NOT NULL,
        category TEXT NOT NULL,
        description TEXT,
        date TEXT
    )
"""

def _save(self, db_path: str = DB_FILENAME) -> None:
    """Save every expense currently in this tracker into the sqlite database at db_path."""
    connection = sqlite3.connect(db_path)   # opens the file (creates it if it doesnt exist yet)
    cursor = connection.cursor()            # a cursor is what we use to actually run SQL statements
    cursor.execute(CREATE_TABLE_SQL)        # make sure the table exists before we try to insert anything
    cursor.execute("DELETE FROM expenses")  # simple approach for this demo: clear old rows first, then re-insert everything fresh
    for expense in self.expenses:
        # using "?" placeholders (NOT f-strings!) to safely insert values - this avoids SQL injection, same idea as lesson 13
        cursor.execute(
            "INSERT INTO expenses (amount, category, description, date) VALUES (?, ?, ?, ?)",
            (expense.amount, expense.category, expense.description, expense.date),
        )
    connection.commit()  # writes the changes to disk, without this our inserts would be lost
    connection.close()   # always close the connection when we are done with it

def _load(self, db_path: str = DB_FILENAME) -> None:
    """Replace this tracker's expenses with whatever is currently stored in the sqlite database at db_path."""
    connection = sqlite3.connect(db_path)
    cursor = connection.cursor()
    cursor.execute(CREATE_TABLE_SQL)  # in case load() is called on a fresh file that was never saved to before
    cursor.execute("SELECT amount, category, description, date FROM expenses")
    rows = cursor.fetchall()          # fetchall() gives us a list of tuples, one tuple per row
    connection.close()
    self.expenses = [Expense(amount=row[0], category=row[1], description=row[2], date=row[3]) for row in rows]  # rebuild our Expense objects

# attaching these two functions as real methods on the ExpenseTracker class we already defined above
# (we could have written them directly inside the class body too, doing it this way just keeps this cell focused on persistence only)
ExpenseTracker.save = _save
ExpenseTracker.load = _load

print("save() and load() are now available on ExpenseTracker")

save() and load() are now available on ExpenseTracker


## step 5: a lazy report generator + a timing decorator (recap generators/decorators, lesson 09, and logging, lesson 15)
* instead of building one giant report string upfront, we write a GENERATOR function that yields one formatted line at a time - this is more memory friendly for big reports and lets the caller stop early if it wants
* we also write a DECORATOR that measures how long producing the report takes and logs it
* important gotcha: if you just call a generator function, python does NOT run its body yet - it only returns a generator object! so our decorator has to be a generator itself too (using `yield from`) to correctly measure the time the ENTIRE report generation takes, including the moment its actually consumed

In [7]:
def timed_report(generator_function):
    """A decorator for GENERATOR functions - times how long it takes until the wrapped generator is fully exhausted."""
    @wraps(generator_function)  # keeps the original functions __name__/__doc__ instead of replacing them with "wrapper"
    def wrapper(*args, **kwargs):
        start = time.perf_counter()  # perf_counter is the recommended clock for measuring short durations, more precise than time.time()
        try:
            # "yield from" delegates to the inner generator - this keeps everything lazy, line by line, instead of collecting it all into a list first
            yield from generator_function(*args, **kwargs)
        finally:
            # this only runs once the generator is fully exhausted (or closed) - exactly what we want to measure "how long the whole report took"
            elapsed = time.perf_counter() - start
            logging.info(f"generating the report took {elapsed:.6f} seconds")
    return wrapper


@timed_report
def generate_report(tracker: "ExpenseTracker") -> Iterator[str]:
    """Lazily yield formatted report lines instead of building one big string upfront."""
    yield "=== Expense Report ==="
    summary = tracker.summary_by_category()
    for category, amount in sorted(summary.items()):  # sorted() so the report always has a stable, predictable order
        yield f"{category:15s}: {amount:8.2f}"
    yield "-" * 27
    yield f"{'TOTAL':15s}: {tracker.total():8.2f}"


# lets try it - note how we loop over generate_report(...), we never build one big string ourselves
for line in generate_report(demo_tracker):
    print(line)

INFO: generating the report took 0.000121 seconds


=== Expense Report ===
food           :    20.40
transport      :    45.00
---------------------------
TOTAL          :    65.40


## a short detour: type hints are documentation, not enforcement (recap lesson 10)
* we put type hints on basically every function/method above (`amount: float`, `-> Expense`, ...)
* these hints help humans (and IDEs, and tools like `mypy`) understand what a function expects and returns
* but python itself never checks them at runtime - passing the "wrong" type does NOT raise a `TypeError` by itself, it will only fail later if the wrong type actually causes a problem somewhere inside the function

In [8]:
def total_of(amounts: list[float]) -> float:
    """Type hints say this wants a list of floats and returns a float - but python does not enforce that at all."""
    return sum(amounts)

print(total_of([1.5, 2.5, 3.0]))   # works fine, matches the type hint

# now lets deliberately break the type hint and pass strings instead of floats
try:
    print(total_of(["1.5", "2.5"]))  # the type hint says list[float], but python happily lets us call it with strings anyway
except TypeError as error:
    # it doesnt fail because we "violated a type hint" - it fails because sum() cant add strings together, a completely normal runtime error
    print("this blew up at RUNTIME, not because of the type hint:", error)

# a real project would catch this kind of mistake ahead of time by running a static type checker like "mypy" over the codebase
# but that is a separate tool you run outside of python itself, not something the interpreter does for you automatically

7.0
this blew up at RUNTIME, not because of the type hint: unsupported operand type(s) for +: 'int' and 'str'


## step 6: a couple of unit tests (recap testing, lesson 15)
* lets write a small `unittest.TestCase` covering the two most important behaviours: that invalid expenses get rejected, and that the category summary math is correct
* we run the tests with `unittest.main(argv=["ignored"], exit=False, verbosity=2)` instead of the normal `unittest.main()` - a plain `unittest.main()` calls `sys.exit()` when it finishes, which would kill our notebook kernel! these extra arguments tell it to behave nicely inside a notebook instead

In [9]:
class TestExpenseTracker(unittest.TestCase):

    def test_negative_amount_raises_invalid_expense_error(self):
        tracker = ExpenseTracker()
        # assertRaises is a context manager - the test only passes if the code inside the "with" block raises exactly this exception
        with self.assertRaises(InvalidExpenseError):
            tracker.add_expense(-1, "food", "should not be allowed", "2024-01-01")

    def test_empty_category_raises_invalid_expense_error(self):
        tracker = ExpenseTracker()
        with self.assertRaises(InvalidExpenseError):
            tracker.add_expense(10, "", "no category given", "2024-01-01")

    def test_summary_by_category_sums_correctly(self):
        tracker = ExpenseTracker()
        tracker.add_expense(10, "food", "lunch", "2024-01-01")
        tracker.add_expense(5, "food", "snack", "2024-01-02")
        tracker.add_expense(20, "transport", "taxi", "2024-01-03")
        summary = tracker.summary_by_category()
        self.assertEqual(summary["food"], 15)       # 10 + 5
        self.assertEqual(summary["transport"], 20)  # just the one entry
        self.assertEqual(tracker.total(), 35)       # everything combined


# argv=["ignored"] fakes a command line so unittest doesnt try to parse OUR notebooks real argv
# exit=False stops it from calling sys.exit() afterwards, which would otherwise kill the jupyter kernel
unittest.main(argv=["ignored"], exit=False, verbosity=2)

test_empty_category_raises_invalid_expense_error (__main__.TestExpenseTracker.test_empty_category_raises_invalid_expense_error) ... 

ok


test_negative_amount_raises_invalid_expense_error (__main__.TestExpenseTracker.test_negative_amount_raises_invalid_expense_error) ... 

ok


test_summary_by_category_sums_correctly (__main__.TestExpenseTracker.test_summary_by_category_sums_correctly) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK


## putting it all together: the final demo
* now lets run through the whole workflow end to end, using pretty much everything we built above
* create a tracker, add several expenses (including one INVALID one we catch on purpose), save it to sqlite, create a completely FRESH tracker, load it back from the database file, and print the report using our decorated generator
* at the very end we clean up the demo database file again, we dont want to leave any generated files behind in the repo

In [10]:
# step 1: create a fresh tracker and add a handful of realistic expenses
tracker = ExpenseTracker()
tracker.add_expense(9.50, "food", "sandwich and a drink", "2024-04-01")
tracker.add_expense(60.00, "transport", "monthly public transport pass", "2024-04-01")
tracker.add_expense(15.20, "food", "groceries top-up", "2024-04-03")
tracker.add_expense(120.00, "rent", "share of the apartment rent", "2024-04-01")
tracker.add_expense(4.30, "food", "coffee", "2024-04-05")

# step 2: try to add a deliberately invalid expense and catch our custom exception, to prove the validation really works
try:
    tracker.add_expense(-50, "entertainment", "this amount is nonsense", "2024-04-06")
except InvalidExpenseError as error:
    print("blocked an invalid expense as expected:", error)

# step 3: persist everything to our little sqlite demo database
tracker.save(DB_FILENAME)
print(f"saved {len(tracker.expenses)} expenses to {DB_FILENAME}")

# step 4: pretend we are a brand new run of the program - a completely fresh tracker, nothing in memory
fresh_tracker = ExpenseTracker()
print("fresh tracker before loading:", fresh_tracker.list_expenses())  # should be empty, []

# step 5: load the previously saved expenses back from disk into this fresh tracker
fresh_tracker.load(DB_FILENAME)
print(f"loaded {len(fresh_tracker.expenses)} expenses back from {DB_FILENAME}")

# step 6: print the category report using our lazy generator + timing decorator combo
print()
for line in generate_report(fresh_tracker):
    print(line)

# step 7: cleanup! we dont want to leave a generated .db file lying around in the repo
os.remove(DB_FILENAME)
print()
print(f"cleaned up {DB_FILENAME}, no leftover files behind")

INFO: generating the report took 0.000064 seconds


blocked an invalid expense as expected: amount must be positive, got -50
saved 5 expenses to capstone_expense_tracker_demo.db
fresh tracker before loading: []
loaded 5 expenses back from capstone_expense_tracker_demo.db

=== Expense Report ===
food           :    29.00
rent           :   120.00
transport      :    60.00
---------------------------
TOTAL          :   209.00

cleaned up capstone_expense_tracker_demo.db, no leftover files behind


## Ideas to extend this project further
* the course ends here, but this project absolutely doesnt have to - here are a few open-ended ideas to keep practicing with, on your own, no solutions provided this time!
* add a simple text-menu CLI loop using `input()` (recap lesson 02/03) so you can add/list/delete expenses interactively from the terminal instead of hardcoding them in a cell
* add monthly (or yearly) filtering, by parsing the `date` field with `datetime.strptime()` and grouping/summing by month instead of only by category (recap lesson 12)
* add a REST API on top with `http.server` (recap lesson 14), exposing endpoints like `GET /expenses` and `POST /expenses` so other programs (or a future frontend) could talk to your tracker over HTTP
* expand the `unittest` suite - cover `filter_by_category`, `save`/`load` round-tripping, edge cases like adding an expense with a weird date format, and maybe add a `setUp`/`tearDown` to avoid repeating the same tracker setup in every test
* add a CSV export feature using the `csv` module (recap lesson 05), so a user could open their expenses in a spreadsheet program
* add a "budget limit" feature: let the user set a monthly budget per category, and have the tracker warn (or raise a new custom exception) when adding an expense would push a category over its limit
* package the whole thing properly as an installable module/package with a `pyproject.toml` (recap lesson 16), so it could be `pip install`-ed and reused in other projects

* congratulation, you finished the whole course from beginner to expert!

## what we learned in this capstone
* combined dataclasses, custom exceptions, OOP, sqlite3 persistence, generators, decorators, logging, type hints and unittest into one small coherent "Personal Expense Tracker" project
* saw a real gotcha with decorating generator functions (`yield from` vs. just calling and returning) and why type hints dont get enforced at runtime
* practiced end to end: validate input -> store in memory -> persist to disk -> reload -> report -> cleanup, the same basic shape most real applications have

## what we learned in the whole course
* 01: how to use jupyter notebooks - cells, markdown, running and re-running code
* 02: python basics - variables, the core data types, strings, lists, dicts and basic operators
* 03: control flow (if/elif/else, while/for loops) and defining our own functions
* 04: advanced functions - *args/**kwargs, variable scope, lambdas, map/filter/reduce, closures and recursion
* 05: modules, packages, working with files, pathlib, and structured data with json/csv
* 06: error handling - try/except/else/finally, raising exceptions, and writing our own custom exception classes
* 07: OOP basics - classes, objects, attributes and methods
* 08: OOP advanced - inheritance, polymorphism and more advanced class features
* 09: iterators, generators and decorators
* 10: typing - type hints and dataclasses
* 11: concurrency - running work in parallel
* 12: working with data - regular expressions, datetime, and the collections module
* 13: databases - storing and querying data with sqlite3
* 14: the web and APIs - talking to and building simple web services
* 15: testing, debugging and logging
* 16: packaging and performance
* 17: python compared to other programming languages
* 18 (this one!): the capstone project, tying it all together into one small real project

thank you for following along all the way to the end - now go build something of your own with it!